# PEBBLES Family EDA — Round 5

Famiglia: `PEBBLES_XS`, `PEBBLES_S`, `PEBBLES_M`, `PEBBLES_L`, `PEBBLES_XL`

Obiettivo: capire se le 5 varianti sono indipendenti, correlate, anticorrelate o legate da una relazione strutturale (basket / synthetic).

**Pipeline**
1. Caricamento prezzi (mid) + trades (day 2,3,4)
2. Statistiche descrittive e prezzi normalizzati
3. Correlazioni (Pearson, Spearman) sui livelli e sui ritorni
4. Rolling correlation
5. Lead-lag via Cross-Correlation Function (CCF)
6. Granger causality
7. Cointegrazione (Engle-Granger pairwise + Johansen multivariato)
8. Regressione XL ~ XS+S+M+L (synthetic basket)
9. Spread Z-score e half-life di mean reversion
10. Verdetto strategico

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path('..').resolve().parent
DATA = ROOT / 'Data_ROUND_5'
OUT  = Path('.').resolve()
OUT.mkdir(exist_ok=True)

FAMILY = 'PEBBLES'
VARIANTS = ['XS','S','M','L','XL']
SYMBOLS = [f'{FAMILY}_{v}' for v in VARIANTS]
DAYS = [2,3,4]
print('Symbols:', SYMBOLS)
print('Data dir:', DATA)

## 1. Loading

In [ ]:
def load_prices(days):
    frames = []
    for d in days:
        f = DATA / f'prices_round_5_day_{d}.csv'
        df = pd.read_csv(f, sep=';')
        df = df[df['product'].isin(SYMBOLS)].copy()
        df['day'] = d
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    out['t'] = out['day']*1_000_000 + out['timestamp']
    return out

def load_trades(days):
    frames = []
    for d in days:
        f = DATA / f'trades_round_5_day_{d}.csv'
        df = pd.read_csv(f, sep=';')
        df = df[df['symbol'].isin(SYMBOLS)].copy()
        df['day'] = d
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

prices = load_prices(DAYS)
trades = load_trades(DAYS)
print(prices.shape, trades.shape)
print(prices['product'].value_counts())

In [ ]:
mid = prices.pivot_table(index='t', columns='product', values='mid_price').sort_index()
mid = mid[SYMBOLS]
mid.to_parquet(OUT/'_mid.parquet')
print(mid.shape)
mid.head()

## 2. Descriptive stats + normalized prices

In [ ]:
desc = mid.agg(['mean','std','min','max']).T
desc['range'] = desc['max'] - desc['min']
desc['cv']    = desc['std']/desc['mean']
desc.to_csv(OUT/'p1_descriptives.csv')
desc.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(13,5))
norm = mid.divide(mid.iloc[0])
for c in norm.columns:
    ax.plot(norm.index, norm[c], label=c, lw=0.8)
ax.axhline(1.0, color='k', lw=0.5, alpha=0.3)
ax.set_title('PEBBLES — mid price normalized to t0=1.0 (days 2-4 concatenated)')
ax.set_xlabel('t (synthetic = day*1e6 + timestamp)')
ax.legend(ncol=5)
fig.tight_layout()
fig.savefig(OUT/'p1_prices_norm.png', dpi=120)
plt.show()

## 3. Correlations — levels & returns

In [ ]:
ret = mid.pct_change().dropna()
ret.to_parquet(OUT/'_ret.parquet')

corr_lvl_p = mid.corr(method='pearson')
corr_lvl_s = mid.corr(method='spearman')
corr_ret_p = ret.corr(method='pearson')
corr_ret_s = ret.corr(method='spearman')

fig, axes = plt.subplots(2,2, figsize=(11,9))
for ax, mat, title in zip(axes.ravel(),
                          [corr_lvl_p, corr_lvl_s, corr_ret_p, corr_ret_s],
                          ['Levels Pearson','Levels Spearman','Returns Pearson','Returns Spearman']):
    im = ax.imshow(mat.values, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(SYMBOLS))); ax.set_xticklabels(VARIANTS)
    ax.set_yticks(range(len(SYMBOLS))); ax.set_yticklabels(VARIANTS)
    for i in range(len(SYMBOLS)):
        for j in range(len(SYMBOLS)):
            ax.text(j, i, f'{mat.values[i,j]:.2f}', ha='center', va='center', fontsize=8)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout(); fig.savefig(OUT/'p3_correlations.png', dpi=120)
plt.show()

corr_lvl_p.to_csv(OUT/'p3_corr_levels_pearson.csv')
corr_ret_p.to_csv(OUT/'p3_corr_returns_pearson.csv')
print('Levels Pearson:'); print(corr_lvl_p.round(3))
print('\nReturns Pearson:'); print(corr_ret_p.round(3))

## 4. Rolling correlation
Per ogni coppia, finestra mobile ⇒ vediamo se la correlazione è stabile o regime-dependent.

In [ ]:
from itertools import combinations
WIN = 500
pairs = list(combinations(SYMBOLS, 2))
fig, axes = plt.subplots(5,2, figsize=(14,12), sharex=True)
axes = axes.ravel()
rcorrs = {}
for i,(a,b) in enumerate(pairs):
    rc = ret[a].rolling(WIN).corr(ret[b])
    rcorrs[f'{a}|{b}'] = rc
    axes[i].plot(rc.index, rc.values, lw=0.7)
    axes[i].axhline(0, color='k', lw=0.4)
    axes[i].set_title(f'{a.split("_")[1]} ↔ {b.split("_")[1]}', fontsize=9)
    axes[i].set_ylim(-1,1)
fig.suptitle(f'Rolling correlation on returns (window={WIN})')
fig.tight_layout(); fig.savefig(OUT/'p3_rolling_corr.png', dpi=120)
plt.show()
pd.DataFrame(rcorrs).to_parquet(OUT/'p3_rolling_corr.parquet')

## 5. Lead-Lag — Cross-Correlation Function
Per ogni coppia, calcola CCF su returns con lag ±50 ticks. Se il picco è a lag>0, A guida B; se a lag<0, B guida A.

In [ ]:
def ccf(x, y, max_lag=50):
    x = (x - x.mean())/x.std()
    y = (y - y.mean())/y.std()
    n = len(x)
    out = []
    for L in range(-max_lag, max_lag+1):
        if L < 0:
            c = np.corrcoef(x.iloc[:L].values, y.iloc[-L:].values)[0,1]
        elif L > 0:
            c = np.corrcoef(x.iloc[L:].values, y.iloc[:-L].values)[0,1]
        else:
            c = np.corrcoef(x.values, y.values)[0,1]
        out.append(c)
    return np.array(out), np.arange(-max_lag, max_lag+1)

MAX_LAG = 30
rows = []
fig, axes = plt.subplots(5,2, figsize=(14,12), sharex=True)
axes = axes.ravel()
for i,(a,b) in enumerate(pairs):
    c, lags = ccf(ret[a], ret[b], MAX_LAG)
    peak = lags[np.argmax(np.abs(c))]
    axes[i].stem(lags, c, basefmt=' ')
    axes[i].axvline(0, color='k', lw=0.5)
    axes[i].axvline(peak, color='r', lw=0.8, ls='--', alpha=0.6)
    axes[i].set_title(f'{a.split("_")[1]} → {b.split("_")[1]}  peak={peak} (r={c[np.argmax(np.abs(c))]:.2f})', fontsize=9)
    rows.append({'a':a,'b':b,'peak_lag':peak,'peak_corr':c[np.argmax(np.abs(c))]})
fig.suptitle(f'Cross-correlation on returns (lag in ticks; peak in red)')
fig.tight_layout(); fig.savefig(OUT/'p4_ccf.png', dpi=120)
plt.show()
ccf_df = pd.DataFrame(rows)
ccf_df.to_csv(OUT/'p4_ccf_peaks.csv', index=False)
ccf_df

## 6. Granger Causality
Un test formale di lead-lag: "i passati di X aiutano a prevedere Y oltre i passati di Y stesso?"

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests
MAX_GLAG = 5
rows = []
for a,b in [(x,y) for x in SYMBOLS for y in SYMBOLS if x!=y]:
    try:
        data = pd.concat([ret[b].rename('y'), ret[a].rename('x')], axis=1).dropna()
        # statsmodels: tests if 2nd column Granger-causes 1st column
        res = grangercausalitytests(data, maxlag=MAX_GLAG, verbose=False)
        for lag in range(1, MAX_GLAG+1):
            p = res[lag][0]['ssr_ftest'][1]
            rows.append({'cause':a,'effect':b,'lag':lag,'pvalue':p})
    except Exception as e:
        rows.append({'cause':a,'effect':b,'lag':None,'pvalue':np.nan,'err':str(e)})
g = pd.DataFrame(rows)
g.to_csv(OUT/'p4_granger.csv', index=False)
# Best lag for each pair
best = g.loc[g.groupby(['cause','effect'])['pvalue'].idxmin()].sort_values('pvalue')
print('Top 10 strongest Granger causalities (lowest p-values):')
best.head(10)

## 7. Cointegration
Engle-Granger pairwise + Johansen su tutta la famiglia. Se troviamo cointegrazione, esiste un equilibrio long-run sfruttabile.

In [ ]:
from statsmodels.tsa.stattools import coint, adfuller
rows = []
for a,b in pairs:
    s, p, _ = coint(mid[a], mid[b])
    rows.append({'a':a,'b':b,'tstat':s,'pvalue':p})
ce = pd.DataFrame(rows).sort_values('pvalue')
ce.to_csv(OUT/'p5_cointegration_pairs.csv', index=False)
print('Engle-Granger pairwise (p<0.05 ⇒ cointegrated):')
ce

In [ ]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen
j = coint_johansen(mid.dropna(), det_order=0, k_ar_diff=1)
trace = j.lr1
cv95  = j.cvt[:,1]
rank_test = pd.DataFrame({'r<=k':[f'r<={k}' for k in range(len(trace))],
                          'trace_stat': trace,
                          'cv_95': cv95,
                          'reject_null': trace > cv95})
rank_test.to_csv(OUT/'p5_johansen.csv', index=False)
print('Johansen trace test:'); print(rank_test)
print('\nFirst eigenvector (cointegration relation):')
evec = pd.Series(j.evec[:,0], index=SYMBOLS)
print(evec.round(4))

## 8. Regression XL ~ XS+S+M+L
Ipotesi: XL è un "basket sintetico" degli altri quattro. Stimo i pesi e guardo il residuo.

In [ ]:
import statsmodels.api as sm
X = mid[[f'{FAMILY}_{v}' for v in ['XS','S','M','L']]]
y = mid[f'{FAMILY}_XL']
Xc = sm.add_constant(X)
model = sm.OLS(y, Xc).fit()
print(model.summary())
with open(OUT/'p6_regression_xl.txt','w') as f:
    f.write(str(model.summary()))
resid = model.resid
adf_p = adfuller(resid)[1]
print(f'\nADF p-value on residuals: {adf_p:.4f}  ⇒ {"stationary" if adf_p<0.05 else "NOT stationary"}')

In [ ]:
fig, axes = plt.subplots(2,1, figsize=(13,7))
axes[0].plot(y.index, y, label='XL actual', lw=0.6)
axes[0].plot(y.index, model.fittedvalues, label='XL fitted (basket of XS+S+M+L)', lw=0.6, alpha=0.8)
axes[0].set_title('XL vs synthetic basket')
axes[0].legend()
axes[1].plot(resid.index, resid, lw=0.5, color='purple')
axes[1].axhline(0, color='k', lw=0.4)
axes[1].set_title(f'Residuals (ADF p={adf_p:.3f})')
fig.tight_layout(); fig.savefig(OUT/'p6_xl_basket.png', dpi=120)
plt.show()

## 9. Z-score and half-life on best cointegrated spread

In [ ]:
def half_life(spread):
    s = spread.dropna()
    s_lag = s.shift(1).dropna()
    s_diff = s.diff().dropna()
    s_lag, s_diff = s_lag.align(s_diff, join='inner')
    beta = np.polyfit(s_lag.values, s_diff.values, 1)[0]
    if beta >= 0: return np.inf
    return -np.log(2)/beta

spread = resid
z = (spread - spread.rolling(500).mean()) / spread.rolling(500).std()
hl = half_life(spread)
print(f'Half-life (XL - basket): {hl:.1f} ticks')

fig, axes = plt.subplots(2,1, figsize=(13,7), sharex=True)
axes[0].plot(spread.index, spread, lw=0.5)
axes[0].set_title(f'XL - basket residual (half-life ≈ {hl:.0f} ticks)')
axes[0].axhline(0, color='k', lw=0.4)
axes[1].plot(z.index, z, lw=0.5, color='darkorange')
for lvl in [-2,-1,1,2]:
    axes[1].axhline(lvl, color='k', lw=0.3, ls='--')
axes[1].set_title('Rolling z-score (window=500)')
fig.tight_layout(); fig.savefig(OUT/'p6_xl_zscore.png', dpi=120)
plt.show()

## 10. Verdetto strategico

In [ ]:
summary = {
    'levels_pearson_XL_vs_XS': corr_lvl_p.loc['PEBBLES_XL','PEBBLES_XS'],
    'returns_pearson_XL_vs_XS': corr_ret_p.loc['PEBBLES_XL','PEBBLES_XS'],
    'best_granger_pvalue': best['pvalue'].iloc[0],
    'best_granger_pair': f"{best['cause'].iloc[0]} -> {best['effect'].iloc[0]}",
    'best_coint_pair': f"{ce['a'].iloc[0]} - {ce['b'].iloc[0]}",
    'best_coint_pvalue': ce['pvalue'].iloc[0],
    'johansen_rank': int(rank_test['reject_null'].sum()),
    'xl_basket_R2': model.rsquared,
    'xl_basket_resid_adf_p': adf_p,
    'xl_basket_halflife_ticks': hl,
}
for k,v in summary.items():
    print(f'{k:35s} {v}')
pd.Series(summary).to_csv(OUT/'verdict.csv')
print('\nAll outputs saved in:', OUT)